# Lecture 30: Linear Regression

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 10) # so that we get shorter print outs of our tables in the Jupyter notebook
set.seed(42) # random seed so your random plots look just like my random plots

## Helper Functions

In [ ]:
# Generate a scatter plot with correlation approximately r
r_scatter <- function(r) {
  n <- 1000
  x <- rnorm(n)
  z <- rnorm(n)
  y <- r * x + sqrt(1 - r^2) * z
  tibble(x = x, y = y) |>
    ggplot(aes(x, y)) +
    geom_point(alpha = 0.4, size = 1) +
    coord_cartesian(xlim = c(-4, 4), ylim = c(-4, 4)) +
    labs(title = paste("r ≈", r)) +
    theme_minimal(base_size = 16)
}

# Generate a tibble of correlated data with correlation approximately r
make_correlated_data <- function(r) {
  x <- rnorm(1000)
  z <- rnorm(1000)
  y <- r * x + sqrt(1 - r^2) * z
  tibble(x = x, y = y)
}

# For a given x value, predict y as the average of nearby y values (within ±0.25)
nn_predict <- function(x_val, df, window = 0.25) {
  neighbors <- df |> filter(x >= x_val - window, x <= x_val + window)
  mean(neighbors$y)
}

## Standard Units and Correlation

The correlation coefficient $r$ is the average of the products of the two variables, when both are measured in standard units.

- $-1 \leq r \leq 1$
- $r = 1$: perfect positive line
- $r = -1$: perfect negative line
- $r = 0$: no linear association (uncorrelated)

In [ ]:
# Note: students will build cor() from scratch in an upcoming assignment
# For now we use R's built-in cor(), which uses sample SD (n-1)
r_scatter(0.99)

In [ ]:
r_scatter(0.5)

In [ ]:
r_scatter(0)

In [ ]:
r_scatter(-0.5)

## Hybrid Cars Data

In [ ]:
hybrid <- read_csv("hybrid.csv", show_col_types = FALSE)
hybrid

In [ ]:
ggplot(hybrid, aes(x = acceleration, y = msrp)) +
  geom_point(alpha = 0.5) +
  labs(
    title = "Hybrid Cars: Acceleration vs. Price",
    x = "Acceleration (km/h/s)",
    y = "MSRP (USD)"
  ) +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$acceleration, hybrid$msrp)

### Switching Axes

$r$ is unaffected by switching which variable is on which axis.

In [ ]:
ggplot(hybrid, aes(x = msrp, y = acceleration)) +
  geom_point(alpha = 0.5) +
  labs(
    title = "Hybrid Cars: Price vs. Acceleration",
    x = "MSRP (USD)",
    y = "Acceleration (km/h/s)"
  ) +
  theme_minimal(base_size = 16)

In [ ]:
cor(hybrid$msrp, hybrid$acceleration)

### Nonlinearity

Correlation only measures *linear* association. A perfect quadratic relationship can have $r = 0$:

In [ ]:
nonlinear <- tibble(
  x = seq(-4, 4, by = 0.5),
  y = x^2
)

ggplot(nonlinear, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  labs(title = "Nonlinear association: y = x²") +
  theme_minimal(base_size = 16)

In [ ]:
cor(nonlinear$x, nonlinear$y)

### Outliers

Outliers can have a big effect on correlation.

In [ ]:
line_data <- tibble(x = 1:4, y = 1:4)

ggplot(line_data, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  labs(title = "Perfect linear data") +
  theme_minimal(base_size = 16)

In [ ]:
cor(line_data$x, line_data$y)

In [ ]:
outlier_data <- tibble(x = 1:5, y = c(1, 2, 3, 4, 0))

ggplot(outlier_data, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  labs(title = "One outlier drastically changes r") +
  theme_minimal(base_size = 16)

In [ ]:
cor(outlier_data$x, outlier_data$y)

### Ecological Correlations

Correlations based on group averages can be misleadingly high. The correlation below is between *state average* SAT scores — not individual students.

In [ ]:
sat2014 <- read_csv("sat2014.csv", show_col_types = FALSE) |>
  arrange(State)
sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math)) +
  geom_point(alpha = 0.5) +
  labs(title = "SAT Scores by State (2014)") +
  theme_minimal(base_size = 16)

In [ ]:
cor(sat2014$`Critical Reading`, sat2014$Math)

In [ ]:
# Categorize states by participation rate
sat2014 <- sat2014 |>
  mutate(
    rate_code = case_when(
      `Participation Rate` <= 25 ~ "low",
      `Participation Rate` <= 75 ~ "medium",
      TRUE                       ~ "high"
    ),
    rate_code = factor(rate_code, levels = c("low", "medium", "high"))
  )
sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math, color = rate_code)) +
  geom_point(size = 2.5, alpha = 0.9) +
  labs(
    title = "SAT 2014: Scores by Participation Rate",
    x = "Critical Reading (state avg)",
    y = "Math (state avg)",
    color = "Participation"
  ) +
  theme_minimal(base_size = 16)

In [ ]:
# Low-participation states — self-selected, high-scoring students
sat2014 |> filter(rate_code == "low")

## Prediction Lines

For data in standard units, what line best predicts y from x?

### r = 0.99

In [ ]:
example <- make_correlated_data(0.99)
head(example, 3)

In [ ]:
ggplot(example, aes(x, y)) +
  geom_point(alpha = 0.5, size = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.99") +
  theme_minimal(base_size = 16)

In [ ]:
# Single nearest-neighbor prediction
nn_predict(-2.25, example)

In [ ]:
# Add nearest-neighbor predictions for every point
example <- example |>
  mutate(predictedY = map_dbl(x, nn_predict, df = example))

In [ ]:
# Graph of averages (gold) vs raw data
ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.99: graph of averages") +
  theme_minimal(base_size = 16)

In [ ]:
# Overlay y = x line — fits very well when r ≈ 1
ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  geom_abline(slope = 1, intercept = 0, color = "#1e90ff", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.99: prediction line y = x") +
  theme_minimal(base_size = 16)

### r = 0

In [ ]:
example <- make_correlated_data(0)

ggplot(example, aes(x, y)) +
  geom_point(alpha = 0.5, size = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0") +
  theme_minimal(base_size = 16)

In [ ]:
example <- example |>
  mutate(predictedY = map_dbl(x, nn_predict, df = example))

# Prediction line is flat (slope = 0) — knowing x tells you nothing about y
ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  geom_abline(slope = 0, intercept = 0, color = "#1e90ff", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0: prediction line is flat") +
  theme_minimal(base_size = 16)

### r = 0.5

In [ ]:
example <- make_correlated_data(0.5)

# Raw scatter with vertical line at x = 1.5
# Most points in that slice are BELOW 1.5, so y = x overestimates
ggplot(example, aes(x, y)) +
  geom_point(alpha = 0.5, size = 1) +
  geom_vline(xintercept = 1.5, color = "black", linewidth = 1) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.5: y = x line (red) overestimates") +
  theme_minimal(base_size = 16)

In [ ]:
example <- example |>
  mutate(predictedY = map_dbl(x, nn_predict, df = example))

# Graph of averages with y = x (red) overlay
ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  geom_vline(xintercept = 1.5, color = "black", linewidth = 1) +
  geom_abline(slope = 1, intercept = 0, color = "red", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.5: graph of averages vs y = x") +
  theme_minimal(base_size = 16)

In [ ]:
# y = r*x (blue) fits the graph of averages; y = x (red) does not
ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  geom_abline(slope = 1,   intercept = 0, color = "red",     linewidth = 1) +
  geom_abline(slope = 0.5, intercept = 0, color = "#1e90ff", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.5: regression line y = 0.5x (blue) vs y = x (red)") +
  theme_minimal(base_size = 16)

### r = 0.7

In [ ]:
example <- make_correlated_data(0.7) |>
  mutate(predictedY = map_dbl(x, nn_predict, df = make_correlated_data(0.7)))

ggplot(example, aes(x = x)) +
  geom_point(aes(y = y), alpha = 0.3, size = 1) +
  geom_point(aes(y = predictedY), color = "gold", size = 1) +
  geom_abline(slope = 1,   intercept = 0, color = "red",        linewidth = 1) +
  geom_abline(slope = 0.7, intercept = 0, color = "dodgerblue", linewidth = 1) +
  coord_cartesian(xlim = c(-3.5, 3.5), ylim = c(-3.5, 3.5)) +
  labs(title = "r = 0.7: regression line y = 0.7x (blue) vs y = x (red)") +
  theme_minimal(base_size = 16)

## Linear Regression: Defining the Line

The regression line in standard units is $\hat{y}_{su} = r \cdot x_{su}$.

Translating back to original units:

$$b_1 = r \cdot \frac{s_y}{s_x}, \qquad b_0 = \bar{y} - b_1 \bar{x}$$

We use `lm()` to fit the regression line. The formula `y ~ x` means "predict y from x".

In [ ]:
# Verify on synthetic r = 0.5 data — slope should be close to 0.5
example <- make_correlated_data(0.5)
lm(y ~ x, data = example)

## Heights Data and Regression Line

In [ ]:
# Child heights are the *adult* heights of children in each family
families <- read_csv("family_heights.csv", show_col_types = FALSE)

heights <- families |>
  mutate(parentAvg = (father + mother) / 2) |>
  select(parentAvg, childHeight)

head(heights, 5)

In [ ]:
# Nearest-neighbor prediction for heights
# Predict child height given parent average, using neighbors within ±0.5 inches
predictChildHeight <- function(h) {
  nearby <- heights |>
    filter(parentAvg >= h - 0.5, parentAvg < h + 0.5)
  mean(nearby$childHeight)
}

In [ ]:
heights_with_pred <- heights |>
  mutate(prediction = map_dbl(parentAvg, predictChildHeight))

head(heights_with_pred, 5)

In [ ]:
# Graph of averages from nearest-neighbor regression
ggplot(heights_with_pred, aes(x = parentAvg)) +
  geom_point(aes(y = childHeight), alpha = 0.3) +
  geom_point(aes(y = prediction), color = "gold", size = 1) +
  labs(
    title = "Predicting Child Height from Parent Average",
    x = "Parent average height",
    y = "Child height"
  ) +
  theme_minimal(base_size = 16)

In [ ]:
# Fit the regression line using lm()
heights_fit <- lm(childHeight ~ parentAvg, data = heights)
heights_fit

In [ ]:
# Add regression predictions using predict()
heights_with_pred <- heights_with_pred |>
  mutate(regressionPrediction = predict(heights_fit, newdata = heights))

heights_with_pred

In [ ]:
# Nearest-neighbor (gold) vs regression line (purple) — very close!
ggplot(heights_with_pred, aes(x = parentAvg)) +
  geom_point(aes(y = childHeight), alpha = 0.3) +
  geom_point(aes(y = prediction), color = "gold", size = 1) +
  geom_line(aes(y = regressionPrediction), color = "#8E2C90", linewidth = 1) +
  labs(
    title = "Heights: Nearest Neighbor vs. Regression Line",
    x = "Parent average height",
    y = "Child height"
  ) +
  theme_minimal(base_size = 16)